In [1]:
import graph_tool.all as gt
from graph_tool.topology import extract_largest_component

import numpy as np
import pandas as pd
import os

import time

from utils.Flow import *

import logging

In [2]:
network_community_info_df = pd.read_csv("network_info_community.csv")

In [3]:
# Filtering all datasets containing at least 1 network containing more than 10_000 vertices

large_networks_df = network_community_info_df[network_community_info_df['num_vertices'] > 10000]
unique_large_networks_ds_names = large_networks_df['ds_name'].unique()

In [4]:
large_networks_df.head()

In [5]:
print(unique_large_networks_ds_names)

In [6]:
new_input_graphs = [
    'academia_edu',
    'amazon_copurchases/302',
    'anybeat',
    'arxiv_authors/CondMat',
    'arxiv_citation/HepPh',
    'as_skitter',
    'baidu',
    'berkstan_web',
    'caida_as/20071112',
    'chicago_road',
    'citeseer',
    'cora',
    'dblp_cite',
    'dblp_coauthor_snap',
    # 'dbpedia_link', ## Way to big
    'douban',
    'ego_social/gplus_combined',
    'email_enron',
    'email_eu',
    'epinions_trust',
    'flickr_aminer',
    'flickr_growth',
    'flixster',
    'foursquare_friendships/new',
    'gnutella/31',
    'google',
    'google_plus',
    'google_web',
    'hyves',
    'inploid',
    'internet_as',
    'lastfm_aminer',
    'linux',
    'livejournal',
    'livemocha',
    'marker_cafe',
    'marvel_universe',
    'mislove_osn/youtube',
    'myspace_aminer',
    'notre_dame_web',
    'openstreetmap/01-AL-counties-street_networks:01073_Jefferson_County', ## weight needs to be parsed
    'petster',
    'pgp_strong',
    'pokec',
    'python_dependency',
    'roadnet/CA',
    'scotus_majority/2008',
    # 'soc_net_comms', # - seems like repettition
    'social_location/brightkite',
    'stanford_web',
    'trec_web',
    'tree-of-life/9606',
    'twitter',
    'twitter_15m',
    # 'twitter_2009', # - really big network
    'twitter_sample',
    # 'twitter_social', # - too big
    'us_patents',
    # 'wiki_users', # -make positive weights
    # 'wikiconflict', # - make weight positive
    'wikipedia-en-talk',
    'wikipedia_growth',
    'wikipedia_link/az',
    'wikitree',
    'wordnet',
    'yahoo_ads',
    'sp_infectious',
    'genetic_multiplex/Homo',
    ('arxiv_collab/cond-mat-2005', 'value'),
    'prosper',
    'wiki_link_dyn',
    ('mist/ppi_interolog_worm', 'Count_paper'),
    'libimseti',
    'twitter_higgs/reply',
    'dblp_coauthor',
    'twitter_events/NYClimateMarch2014',
    'qa_user/askubuntu_all',
    'mag_history_coauthor/full-proj',
    'mag_geology_coauthor/full-proj',
    'wiki_talk/de',
    'dbpedia_all',
    'dblp_simplices/full-proj',
    ('bitcoin', 'count'),
    'word_adjacency/spanish',
    'facebook_wall',
    'digg_reply',
    'foldoc',
    'fly_hemibrain',
    'lkml_reply',
    'cofe',
    'slashdot_threads',
    'word_assoc',
    'human_brains/BNU1_0025915_2_DTI_DS16784',
    ('us_agencies/aggregate', 'link_counts'),
    'physics_collab/arXiv',
    'topology',
    'us_roads/DE'
]

In [6]:
graph_name = 'yahoo_ads'

graph = gt.collection.ns[graph_name]

# display(graph.list_properties())

display(graph.edge_properties)

display(network_community_info_df[network_community_info_df['name'] == graph_name]['edge_properties'])

In [ ]:
def is_graph_weighted(graph, weight_key):
    if weight_key not in graph.edge_properties:
        return False
    
    eprop = graph.edge_properties[weight_key]

    unique_values, counts = np.unique(eprop.get_array(), return_counts=True)
    logging.info(f'Graph has {counts} unique values: {unique_values}')

    return len(unique_values) >= 4

In [ ]:
def get_mst(graph, edge_property = None):
    if(edge_property is None):
        weight_key = "weight"
    else:
        weight_key = edge_property
    
    is_weighted = is_graph_weighted(graph, weight_key)

    if (weight_key in graph.edge_properties 
        and is_weight_type_supported(graph.ep[weight_key].value_type()) 
        and is_weighted):

        graph.ep[weight_key].a *= -1
    else:
        if is_weighted:
            logging.error("No supported weights found")
        else:
            logging.error("Unweighted network - can not extract MST")
        return False
    
    try:
        mst_tree = gt.min_spanning_tree(graph, graph.ep[weight_key])
        return gt.GraphView(graph, efilt=mst_tree)
    
    except Exception as ex:
        msg = f"Error during mst extraction: {ex}"
        print(msg)
        logging.error(msg)
        return False

In [7]:
def prepare_graph(graph_name, weight_key=None):
    path = prepare_folder(graph_name)

    graph = gt.collection.ns[graph_name]
    if (has_parallel_edges(graph)):
        graph, _ = simplify_graph_with_weights(
            graph=graph, weight_name=weight_key, inplace=True)

    if (graph.is_directed()):
        graph, _ = directed_to_undirected_sum_weights(
            g_directed=graph, existing_weight_name=weight_key, inplace=True)

    largest_component = extract_largest_component(
        graph, directed=False, prune=True)
    
    file_path = get_network_file_path(graph_name, path, False)

    largest_component.save(file_path)
    
    mst = get_mst(largest_component, weight_key)

    if mst:
        mst_path = get_network_file_path(graph_name, path, True)
        mst.save(mst_path)

In [ ]:
def run_graph_prep_flow(data):
    config_logging(into="graph_prep_log.log")

    logging.info(f"Starting preparation for {len(data)} graphs...")
    start_time = time.time()

    for graph_data in data:
        if isinstance(graph_data, tuple) and len(graph_data) >= 2:
            in_separate_process(
                run = prepare_graph,
                withArgs=(graph_data[0], graph_data[1]),
                log_as=graph_data[0]
            )
        else:
            in_separate_process(
                run=prepare_graph,
                withArgs=(graph_data,),
                log_as=graph_data
            )

    end_time = time.time()

    computation_time = end_time - start_time

    readable_time = time.strftime("%H:%M:%S", time.gmtime(computation_time))

    logging.info(f"Graph preparation finished at {end_time} and took: {readable_time}")

    return computation_time

In [9]:
compute_time = run_graph_prep_flow(new_input_graphs)

In [10]:
compute_time/60/60